# Von Mises Plasticity — Interactive Notebook

## 1. Brief background

Most **ductile metals** yield according to the **von Mises criterion** (also called the
$J_2$ or *distortion-energy* criterion). The idea: a material starts to yield when the
**distortional** (shape-changing) strain energy reaches a critical value. Hydrostatic
pressure (equal squeezing in all directions) only changes volume, not shape, so it does
**not** cause yielding.

**Von Mises equivalent stress** (a single scalar that summarizes a full stress state):

$$\sigma_\text{vm}=\sqrt{\tfrac{1}{2}\left[(\sigma_1-\sigma_2)^2+(\sigma_2-\sigma_3)^2+(\sigma_3-\sigma_1)^2\right]}$$

or, in terms of the stress components,

$$\sigma_\text{vm}=\sqrt{\tfrac{1}{2}\left[(\sigma_{xx}-\sigma_{yy})^2+(\sigma_{yy}-\sigma_{zz})^2+(\sigma_{zz}-\sigma_{xx})^2\right]+3\left(\tau_{xy}^2+\tau_{yz}^2+\tau_{zx}^2\right)}$$

**Yield criterion:**

$$f=\sigma_\text{vm}-\sigma_Y \;\;\begin{cases}<0 & \text{elastic (inside the surface)}\\[2pt] =0 & \text{yielding (on the surface)}\\[2pt] >0 & \text{not admissible (outside)}\end{cases}$$

where $\sigma_Y$ is the current yield stress of the material.

### Plane-stress cross-section (the 2D graph)

In principal-stress space the von Mises surface is a **cylinder** whose axis is the
hydrostatic line $\sigma_1=\sigma_2=\sigma_3$. Taking a **plane-stress** slice
($\sigma_3=0$) gives an **ellipse**:

$$\sigma_1^2-\sigma_1\sigma_2+\sigma_2^2=\sigma_Y^2$$

Its major axis lies along $\sigma_1=\sigma_2$ (semi-axis $\sqrt{2}\,\sigma_Y$) and its
minor axis along $\sigma_1=-\sigma_2$ (semi-axis $\sqrt{2/3}\,\sigma_Y$). That ellipse is
what we plot and explore below.

### Hardening (why the sliders change the surface)

For a perfectly-plastic material the surface is fixed. With **isotropic hardening** the
surface *expands* as the material accumulates plastic strain $\varepsilon_p$:

$$\sigma_Y=\sigma_{y0}+H\,\varepsilon_p$$

where $\sigma_{y0}$ is the initial yield stress and $H$ is the (linear) hardening
modulus. The sliders below let you vary $\sigma_{y0}$, $H$ and $\varepsilon_p$ to see the
ellipse grow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 2. Helper functions

* `von_mises_plane_stress` — equivalent stress for a 2D (plane-stress) state
  $(\sigma_{xx},\sigma_{yy},\tau_{xy})$:
  $\;\sigma_\text{vm}=\sqrt{\sigma_{xx}^2-\sigma_{xx}\sigma_{yy}+\sigma_{yy}^2+3\tau_{xy}^2}$
* `principal_stresses` — Mohr's-circle principal values so we can plot the point in the
  $(\sigma_1,\sigma_2)$ plane.
* `yield_ellipse` — points on the von Mises ellipse for a given yield stress.

In [ ]:
def von_mises_plane_stress(sxx, syy, txy):
    """Von Mises equivalent stress for a plane-stress state (MPa)."""
    return np.sqrt(sxx**2 - sxx*syy + syy**2 + 3.0*txy**2)


def principal_stresses(sxx, syy, txy):
    """In-plane principal stresses from Mohr's circle."""
    avg = 0.5*(sxx + syy)
    R = np.sqrt((0.5*(sxx - syy))**2 + txy**2)
    return avg + R, avg - R


def yield_ellipse(sigma_Y, n=400):
    """Points (s1, s2) on the von Mises plane-stress yield ellipse."""
    t = np.linspace(0.0, 2.0*np.pi, n)
    u = np.sqrt(2.0) * sigma_Y * np.cos(t)      # along the sigma1 = sigma2 axis
    v = np.sqrt(2.0/3.0) * sigma_Y * np.sin(t)  # along the sigma1 = -sigma2 axis
    s1 = (u + v)/np.sqrt(2.0)
    s2 = (u - v)/np.sqrt(2.0)
    return s1, s2

## 3. Interactive yield surface + stress-point checker

Use the sliders to:

* **Material properties** — change the initial yield stress $\sigma_{y0}$, the hardening
  modulus $H$, and the accumulated plastic strain $\varepsilon_p$. The current yield
  stress is $\sigma_Y=\sigma_{y0}+H\,\varepsilon_p$ and the ellipse resizes accordingly.
* **Stress point** — enter the plane-stress components $\sigma_{xx},\sigma_{yy},\tau_{xy}$.
  The notebook computes the principal stresses, plots the point at $(\sigma_1,\sigma_2)$,
  computes $\sigma_\text{vm}$, and tells you whether the state is **inside (elastic, green)**
  or **outside (yielding, red)** the surface — both numerically and graphically.

The title reports $\sigma_\text{vm}$, the status, the principal stresses, and the
**safety factor** $\sigma_Y/\sigma_\text{vm}$ (>1 means still elastic).

In [ ]:
def plot_von_mises(sigma_y0=250.0, H=0.0, eps_p=0.0,
                    sxx=180.0, syy=-90.0, txy=60.0):
    # current (hardened) yield stress
    sigma_Y = sigma_y0 + H*eps_p

    s1e, s2e = yield_ellipse(sigma_Y)
    s1, s2 = principal_stresses(sxx, syy, txy)
    svm = von_mises_plane_stress(sxx, syy, txy)
    inside = svm <= sigma_Y
    color = 'seagreen' if inside else 'crimson'

    fig, ax = plt.subplots(figsize=(7, 7))

    # yield surface
    ax.plot(s1e, s2e, color='steelblue', lw=2.2,
            label=f'Yield surface  $\\sigma_Y$={sigma_Y:.0f} MPa')
    ax.fill(s1e, s2e, color='steelblue', alpha=0.08)

    # reference axes / hydrostatic line
    lim = max(np.max(np.abs(np.concatenate([s1e, s2e])))*1.15,
              abs(s1)*1.2, abs(s2)*1.2, 1.0)
    ax.axhline(0, color='0.6', lw=0.8)
    ax.axvline(0, color='0.6', lw=0.8)
    ax.plot([-lim, lim], [-lim, lim], '--', color='0.7', lw=0.8,
            label='hydrostatic axis  $\\sigma_1=\\sigma_2$')

    # stress point
    ax.plot([0, s1], [0, s2], ls=':', color=color, lw=1.2)
    ax.plot(s1, s2, 'o', color=color, ms=12, zorder=5,
            markeredgecolor='k', markeredgewidth=0.6)
    ax.annotate(f'  ({s1:.0f}, {s2:.0f})', (s1, s2), color=color,
                fontsize=10, va='center')

    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_xlabel(r'$\sigma_1$  (MPa)')
    ax.set_ylabel(r'$\sigma_2$  (MPa)')
    ax.grid(alpha=0.3)
    ax.legend(loc='upper left', fontsize=9)

    status = 'INSIDE  (elastic)' if inside else 'OUTSIDE  (yielding)'
    sf = sigma_Y/svm if svm > 0 else np.inf
    ax.set_title(
        f'$\\sigma_{{vm}}$ = {svm:.1f} MPa   $\\rightarrow$   {status}\n'
        f'principal: $\\sigma_1$={s1:.1f}, $\\sigma_2$={s2:.1f} MPa'
        f'    |    safety factor = {sf:.2f}',
        color=color, fontsize=12)
    plt.show()


interact(
    plot_von_mises,
    sigma_y0=FloatSlider(value=250, min=50,  max=600, step=10,
                         description='σ_y0 (MPa)', style={'description_width':'initial'}),
    H=FloatSlider(value=0, min=0, max=5000, step=100,
                  description='Hardening H (MPa)', style={'description_width':'initial'}),
    eps_p=FloatSlider(value=0, min=0, max=0.10, step=0.005, readout_format='.3f',
                      description='plastic strain ε_p', style={'description_width':'initial'}),
    sxx=FloatSlider(value=180, min=-600, max=600, step=10,
                    description='σ_xx (MPa)', style={'description_width':'initial'}),
    syy=FloatSlider(value=-90, min=-600, max=600, step=10,
                    description='σ_yy (MPa)', style={'description_width':'initial'}),
    txy=FloatSlider(value=60, min=-400, max=400, step=10,
                    description='τ_xy (MPa)', style={'description_width':'initial'}),
);

## 4. Quick numeric check (no sliders)

If you just want to test a single stress state against a fixed yield stress, edit the
values below and run the cell.

In [ ]:
# --- edit these ---
sigma_Y = 250.0          # yield stress (MPa)
sxx, syy, txy = 200.0, -100.0, 80.0   # plane-stress components (MPa)
# ------------------

svm = von_mises_plane_stress(sxx, syy, txy)
s1, s2 = principal_stresses(sxx, syy, txy)

print(f'Principal stresses : sigma_1 = {s1:8.2f} MPa,  sigma_2 = {s2:8.2f} MPa')
print(f'Von Mises stress   : sigma_vm = {svm:7.2f} MPa')
print(f'Yield stress       : sigma_Y  = {sigma_Y:7.2f} MPa')
print(f'Safety factor      : {sigma_Y/svm:6.3f}')
print('Result             :',
      'INSIDE  -> elastic' if svm <= sigma_Y else 'OUTSIDE -> yielding')